In [5]:
import pandas as pd

In [6]:
tc = pd.read_csv('IHME_USA_HEALTH_CARE_SPENDING_CAUSE_COUNTY_2010_2019_AGGREGATE_TABLES/IHME_USA_HEALTH_CARE_SPENDING_CAUSE_COUNTY_2010_2019_TABLE_6_NATIONAL_STATE_CONDITION_2010_2019_Y2025M02D13.csv')
tc = tc[(tc['location_name']=='United States') & (tc['year_id']==2019) & (tc['age_name']=='All ages')].reset_index(drop=True)[['year_id', 'location_name', 'acause', 'cause_name', 'spend_mean']]
tc_ihd = tc[tc['acause']=='cvd_ihd'].reset_index(drop=True)
tc_is = tc[tc['acause']=='cvd_stroke'].reset_index(drop=True)
tc_pad = tc[tc['acause']=='cvd_pvd'].reset_index(drop=True)

tc_is['spend_mean'] = tc_is['spend_mean'] * 0.8534585065
tc_is['acause'] = 'cvd_is'
tc_is['cause_name'] = 'Ischemic stroke'

tc_ihd.to_csv('TC_US_IHD.csv', index=False)
tc_is.to_csv('TC_US_IS.csv', index=False)
tc_pad.to_csv('TC_US_PAD.csv', index=False)

tc_usa = pd.concat([tc_ihd, tc_is, tc_pad], ignore_index=True)
tc_usa = tc_usa[['cause_name', 'spend_mean']]
tc_usa.rename(columns={'cause_name':'disease', 'spend_mean':'TC_USA'}, inplace=True)

tc_usa.to_csv('TC_USA.csv', index=False)
tc_usa

,disease,TC_USA
0,Ischemic heart disease,8.066205e+10
1,Ischemic stroke,3.735673e+10
2,Lower extremity peripheral arterial disease,9.351102e+09


In [7]:
## MAIN FUNCTION - input
def input_function(disease):
    ## INPUT FILE
    PPP = True
    # PPP = False
    if PPP:
        df_input_he = pd.read_csv('hepc_ppp.csv')
    else:
        df_input_he = pd.read_csv('hepc.csv')
        
    df_input_prev = pd.read_csv(f'../../data/ASCVD/{disease}/Prevalence_val.csv')
    df_input_prev = df_input_prev[df_input_prev['sex_name']=='B'].reset_index(drop=True)

    year_cols = [c for c in df_input_prev.columns if c.isdigit()]

    prev = (
        df_input_prev
        .groupby('ISO3', as_index=False)[year_cols]
        .sum()
    )

    prev['age_name'] = 'All Ages'
    prev = prev[['ISO3', 'age_name'] + year_cols]

    df_input_USA_TC = pd.read_csv(f'TC_US_{disease.replace(" ", "_")}.csv')
    df_input_USA_TC = df_input_USA_TC[['cause_name', 'spend_mean']]
    df_input_USA_TC.rename(columns={'cause_name':'disease', 'spend_mean':'TC_USA'}, inplace=True)

    he = df_input_he[['Country Code', '2019']]
    he = he.set_index("Country Code")
    he = he.rename(columns = {'2019':'house expend'})
    he_merge = he.merge(prev, left_on="Country Code", right_on="ISO3")

    he_merge = he_merge[["ISO3", "house expend"]]

    data = he_merge.set_index("ISO3")
    prev = prev.set_index("ISO3")

    if disease == 'IHD':
        disease = 'Ischemic heart disease'
    elif disease == 'IS':
        disease = 'Ischemic stroke'
    elif disease == 'PAD':
        disease = 'Lower extremity peripheral arterial disease'
        
    TC_USA_raw = df_input_USA_TC.loc[
        df_input_USA_TC['disease'] == disease, 'TC_USA'
    ].iloc[0]

    prev_use = prev[prev['age_name'] == 'All Ages'].copy()
    year_col = '2019' if '2019' in prev_use.columns else 2019

    usa_prev = prev_use.loc['USA', year_col]
    const = (TC_USA_raw / 2100.1) / usa_prev

    common_idx = data.index.intersection(prev_use.index)
    data.loc[common_idx, disease] = (
        const * prev_use.loc[common_idx, year_col] * data.loc[common_idx, 'house expend']
    )

    data[disease] = data[disease]/1000000000
    
    return data

In [8]:
for disease in ['IHD', 'IS', 'PAD']:
    data = input_function(disease)
    data.to_csv(f'TC_{disease}.csv', index=True)